# Moment-based signal inference — analytical R=0 results

Loads the pickle written by `python-scripts/moment_inference.py` (Section 5 of
`notes/image_likelihood.tex`, "Moment-Based Cloud Marginalisation").

**Analytical (R=0) pipeline** — no importance sampling:
1. Reduce each shot's port-summed image to moments (N_tot, mu_xf, mu_yf, var_xf, var_yf), per AI.
2. Analytic ballistic Kalman with **R=0** (zero measurement error): treat sample moments as exact.
   The R=0 constraint projects the prior mean onto the 1D ballistic constraint mu_x0+T·mu_vx0=mu_xf,
   yielding a unique point estimate eta_hat per (shot, AI).  The 8D nuisance integral reduces to
   a single point evaluation — no sampling needed.
3. Pre-compute pixel ACS tensors at eta_hat for every (shot, AI) pair on GPU.
4. For a trial beta=(As,Ac), rotate the Z100 ACS by delta_phi_i(beta), evaluate the
   phi0-marginalised Poisson image logL via logsumexp over phi grid, sum shots.
5. Optimise / grid-scan logL(beta) over (As, Ac).

**Run the script in a terminal first:**
```bash
mkdir -p logs results
nohup python python-scripts/moment_inference.py > logs/moment_inference.log 2>&1 &
tail -f logs/moment_inference.log
```

In [ ]:
import pickle, os
import numpy as np
import matplotlib.pyplot as plt

REPO = os.path.expanduser(
    '~/aispp-sims/gaussian-wavefront-spatially-resolved-inference')

RESULTS_FILE = os.path.join(REPO, 'results', 'moment_inference.pkl')

with open(RESULTS_FILE, 'rb') as f:
    data = pickle.load(f)

diag_rows  = data['diag_rows']
shot_ids   = data['shot_ids']
f_signal   = data['f_signal']
As_true    = data['As_true']
Ac_true    = data['Ac_true']
beta_hat   = data['beta_hat']
logL_hat   = data['logL_hat']
logL_true  = data.get('logL_true', float('nan'))
logL_zero  = data.get('logL_zero', float('nan'))
grid_ax    = data['grid_ax']
logL_grid  = data['logL_grid']
config     = data['config']

print(f"n_shots used: {len(shot_ids)}")
print(f"config: {config}")
print(f"True beta:      As={As_true:+.4f}  Ac={Ac_true:+.4f}"
      f"  (amp={data['signal_amp_true']:.3f}, phase={data['signal_phase_true']:.3f})")
print(f"Recovered beta: As={beta_hat[0]:+.4f}  Ac={beta_hat[1]:+.4f}  logL={logL_hat:.2f}")
print(f"logL(true)={logL_true:.2f}  logL(0,0)={logL_zero:.2f}  delta={logL_true-logL_zero:.2f}")
print(f"opt_success={data['opt_success']}  msg={data['opt_message']}")

## logL(beta) grid scan

Contour of the phi0-marginalised logL over the (As, Ac) plane,
evaluated at the R=0 Kalman point estimates eta_hat (one per shot per AI).
True beta and the optimiser's beta_hat are marked.
The peak should lie near the true (red) marker when the moment-based
point estimate captures enough of the nuisance variation.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
X, Y = np.meshgrid(grid_ax, grid_ax, indexing='ij')
cs = ax.contourf(X, Y, logL_grid - logL_grid.max(), levels=30, cmap='viridis')
fig.colorbar(cs, ax=ax, label='logL - max(logL)')
ax.plot(As_true, Ac_true, 'r*', ms=18, mec='k', label='true beta')
ax.plot(beta_hat[0], beta_hat[1], 'wx', ms=12, mew=2.5, label='beta_hat')
ax.set_xlabel('As')
ax.set_ylabel('Ac')
n_shots = len(shot_ids)
ax.set_title(f'logL(As, Ac)  ({n_shots} shots, analytical R=0, n_theta={config["n_theta"]})')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## R=0 Kalman point estimate vs. true initial cloud params

With R=0 the Kalman posterior is a delta function at the minimum-prior-cost point on the
ballistic constraint manifold mu_x0+T·mu_vx0=mu_xf.  This scatter plot checks how well
eta_hat recovers the true simulated cloud parameters for both AIs.

Concretely: with a Gaussian prior diag(tau_pos², tau_vel²) and measurement h^T eta=z (exact),
the posterior mean is:

    eta_hat = (tau_pos²/(tau_pos²+T²tau_vel²)) * z  [for mu_x0 component]
    dot_eta_hat = T*tau_vel²/(tau_pos²+T²tau_vel²) * z  [for mu_vx0 component]

When tau_pos=tau_vel=tau (equal priors), this simplifies to mu_x0_hat=z/(1+T²), mu_vx0_hat=T·z/(1+T²).
The true value is constrained to the same line but at a different position along it — bias
arises because the prior is centered at zero, not at the true initial condition.

In [ ]:
# eta_hat ordering: [mu_x0, mu_y0, mu_vx0, mu_vy0, sx0, sy0, svx0, svy0]
# theta_true ordering (from meta): [mu_x0, mu_y0, mu_vx0, mu_vy0, sigma_x, sigma_y, sigma_vx, sigma_vy]
mu_x0_hat_z0  = np.array([r['eta0_hat'][0] for r in diag_rows])
mu_vx0_hat_z0 = np.array([r['eta0_hat'][2] for r in diag_rows])
mu_x0_true_z0  = np.array([r['theta_true_z0'][0] for r in diag_rows])
mu_vx0_true_z0 = np.array([r['theta_true_z0'][2] for r in diag_rows])

mu_x0_hat_z100  = np.array([r['eta1_hat'][0] for r in diag_rows])
mu_vx0_hat_z100 = np.array([r['eta1_hat'][2] for r in diag_rows])
mu_x0_true_z100  = np.array([r['theta_true_z100'][0] for r in diag_rows])
mu_vx0_true_z100 = np.array([r['theta_true_z100'][2] for r in diag_rows])

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
panels = [
    (axes[0, 0], mu_x0_true_z0 * 1e6,  mu_x0_hat_z0 * 1e6,  'mu_x0 [um]  Z0'),
    (axes[0, 1], mu_vx0_true_z0 * 1e6, mu_vx0_hat_z0 * 1e6, 'mu_vx0 [um/s]  Z0'),
    (axes[1, 0], mu_x0_true_z100 * 1e6,  mu_x0_hat_z100 * 1e6,  'mu_x0 [um]  Z100'),
    (axes[1, 1], mu_vx0_true_z100 * 1e6, mu_vx0_hat_z100 * 1e6, 'mu_vx0 [um/s]  Z100'),
]
for ax, t, p, label in panels:
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.6, label='ideal')
    ax.scatter(t, p, s=18, color='steelblue', alpha=0.7)
    ax.set_xlabel('true')
    ax.set_ylabel('R=0 Kalman point estimate')
    ax.set_title(label)
    ax.legend()
plt.suptitle('R=0 Kalman eta_hat vs. true cloud params (x-axis, per shot)')
plt.tight_layout()
plt.show()

## Injected signal delta_phi_i: true vs. recovered model

In [ ]:
shot_idx_arr = np.array(shot_ids, dtype=np.float64)
dphi_true = np.array([r['delta_phi_true'] for r in diag_rows])
dphi_hat = (beta_hat[0] * np.sin(2 * np.pi * f_signal * shot_idx_arr) +
            beta_hat[1] * np.cos(2 * np.pi * f_signal * shot_idx_arr))
dphi_true_model = (As_true * np.sin(2 * np.pi * f_signal * shot_idx_arr) +
                   Ac_true * np.cos(2 * np.pi * f_signal * shot_idx_arr))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(shot_idx_arr, dphi_true, 'o', ms=4, color='k', label='delta_phi (true, stored)')
ax.plot(shot_idx_arr, dphi_true_model, '-', color='g', alpha=0.6, label='true model As,Ac')
ax.plot(shot_idx_arr, dphi_hat, '-', color='r', alpha=0.8, label='recovered beta_hat')
ax.set_xlabel('shot index i')
ax.set_ylabel('delta_phi_i [rad]')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
shot_idx_arr = np.array(shot_ids, dtype=np.float64)
dphi_true = np.array([r['delta_phi_true'] for r in diag_rows])
dphi_hat = beta_hat[0] * np.sin(2 * np.pi * f_signal * shot_idx_arr) + \
           beta_hat[1] * np.cos(2 * np.pi * f_signal * shot_idx_arr)
dphi_true_model = As_true * np.sin(2 * np.pi * f_signal * shot_idx_arr) + \
                  Ac_true * np.cos(2 * np.pi * f_signal * shot_idx_arr)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(shot_idx_arr, dphi_true, 'o', ms=4, color='k', label='delta_phi (true, stored)')
ax.plot(shot_idx_arr, dphi_true_model, '-', color='g', alpha=0.6, label='true model As,Ac')
ax.plot(shot_idx_arr, dphi_hat, '-', color='r', alpha=0.8, label='recovered beta_hat')
ax.set_xlabel('shot index i')
ax.set_ylabel('delta_phi_i [rad]')
ax.legend()
plt.tight_layout()
plt.show()